# DataLake - Jogos Olímpicos
- Aluno: Pedro Yutaro Mont Morency Nakamura
- Disciplina: Ciência de Dados
- Data: Março de 2026

## Descrição da Atividade:
Este notebook utiliza dados históricos das Olimpíadas (1896–2022) e dados
dos Jogos Olímpicos de Paris 2024 para consolidar o total de medalhas
por país, por meio da arquitetura por zonas.

Os dados foram obtidos das seguintes fontes:

- Base dos Dados – Histórico das Olimpíadas
- Kaggle – Paris 2024 Olympic Summer Games

## Objetivo DESSA Pipeline:
1. Limpeza + normalização de arquivos .csv em BRONZE
2. Geração de arquivos .parquet + metadados para a camada SILVER 
3. Fazer o JOIN das fontes e salvar como parquet na camada SILVER

### Integrações (JOIN) a serem feitas:
- Atletas e Medalhas (Para análise de Gênero/Modalidade):
  - Fonte Histórica: Olympic_Athlete_Event_Results.csv + Olympic_Athlete_Bio.csv.
  - Fonte Paris 2024: medallists.csv + athletes.csv.
- Quadro de Medalha (Histórico + Paris 2024) total por estação:
  - Fonte Histórica: Olympic_Games_Medal_Tally.csv + Olympics_Games.csv.
  - Fonte Paris 2024: medals_total.csv.

## 1. Importações e Configurações:

#### Importações Default:

In [1]:
import pandas as pd
import sys
from pathlib import Path

#### Configuração de Path:

In [2]:
root = Path().absolute().parent.parent
sys.path.append(str(root))

#### Importações de módulos locais:

In [3]:
from src import CatalogManager
from src.pipeline_utils import save_and_catalog

#### Catálogo e rotas locais

In [4]:
catalog = CatalogManager()

In [5]:
silver_in = root / "silver"
silver_out = root / "silver" / "joins"

## 2. Integração do Quadro de Medalhas

In [8]:
src_hist = "../../silver/olympics_history/Olympic_Games_Medal_Tally.parquet"
src_paris = "../../silver/olympics_paris/medals_total.parquet"

df_hist = pd.read_parquet(src_hist)
df_paris = pd.read_parquet(src_paris)

In [9]:
print("<===== df_hist =====>")
df_hist.info()

<===== df_hist =====>
<class 'pandas.DataFrame'>
RangeIndex: 1807 entries, 0 to 1806
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   edition      1807 non-null   str  
 1   edition_id   1807 non-null   int64
 2   year         1807 non-null   int64
 3   country      1807 non-null   str  
 4   country_noc  1807 non-null   str  
 5   gold         1807 non-null   int64
 6   silver       1807 non-null   int64
 7   bronze       1807 non-null   int64
 8   total        1807 non-null   int64
 9   season       1807 non-null   str  
dtypes: int64(6), str(4)
memory usage: 208.1 KB


In [10]:
print("<===== df_paris =====>")
df_paris.info()

<===== df_paris =====>
<class 'pandas.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   country_code  92 non-null     str  
 1   country       92 non-null     str  
 2   country_long  92 non-null     str  
 3   gold          92 non-null     int64
 4   silver        92 non-null     int64
 5   bronze        92 non-null     int64
 6   total         92 non-null     int64
dtypes: int64(4), str(3)
memory usage: 6.9 KB


#### Averiguando Mismatch de colunas

In [11]:
diff_1_2 = set(df_hist.columns) - set(df_paris.columns)
print(f"Colunas apenas em df_hist: {diff_1_2}\n")

diff_2_1 = set(df_paris.columns) - set(df_hist.columns)
print(f"Colunas apenas em df_paris: {diff_2_1}")

Colunas apenas em df_hist: {'edition_id', 'country_noc', 'year', 'edition', 'season'}

Colunas apenas em df_paris: {'country_long', 'country_code'}


### 2.1. Renomeando e Populando campos novos

In [20]:
df_paris_aligned = df_paris.rename(columns={
    'country_code': 'country_noc',
})

In [23]:
df_paris_aligned['year'] = 2024
df_paris_aligned['season'] = 'Summer'
df_paris_aligned['edition'] = '2024 Summer Olympics'
df_paris_aligned['edition_id'] = df_hist['edition_id'].max() + 1

In [24]:
df_paris_aligned.info()

<class 'pandas.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   country_noc   92 non-null     str  
 1   country       92 non-null     str  
 2   country_long  92 non-null     str  
 3   gold          92 non-null     int64
 4   silver        92 non-null     int64
 5   bronze        92 non-null     int64
 6   total         92 non-null     int64
 7   year          92 non-null     int64
 8   season        92 non-null     str  
 9   edition       92 non-null     str  
 10  edition_id    92 non-null     int64
dtypes: int64(6), str(5)
memory usage: 12.2 KB


## 3. Concatenando datasets

In [25]:
common = df_paris.columns.intersection(df_hist.columns).tolist()
paris_cols = df_paris.columns.tolist()

print(f"Comum entre paris e hist: {common} --- ({len(common)})")
print(f"paris: {paris_cols} --- ({len(paris_cols)})")

Comum entre paris e hist: ['country', 'gold', 'silver', 'bronze', 'total'] --- (5)
paris: ['country_code', 'country', 'country_long', 'gold', 'silver', 'bronze', 'total'] --- (7)


In [28]:
cols = ['edition_id', 'edition', 'year', 'season', 'country', 'country_noc', 'gold', 'silver', 'bronze', 'total']

df_integrated = pd.concat([
  df_hist[cols],
  df_paris_aligned[cols]
], ignore_index=True)

In [29]:
df_integrated.info()

<class 'pandas.DataFrame'>
RangeIndex: 1899 entries, 0 to 1898
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   edition_id   1899 non-null   int64
 1   edition      1899 non-null   str  
 2   year         1899 non-null   int64
 3   season       1899 non-null   str  
 4   country      1899 non-null   str  
 5   country_noc  1899 non-null   str  
 6   gold         1899 non-null   int64
 7   silver       1899 non-null   int64
 8   bronze       1899 non-null   int64
 9   total        1899 non-null   int64
dtypes: int64(6), str(4)
memory usage: 218.7 KB


#### Salvando JOIN na silver

In [35]:
save_and_catalog(
  filename='integrated_medal_tally_1896_2024',
  file_format='.parquet',
  description='Quadro de medalhas unificado de todas as edições (Histórico + Paris 2024)',
  observations='JOIN feito na 3º etapa da pipeline, com destino ao repositório de joins em silver.',
  df=df_integrated,
  catalog_manager=catalog,
  src=f"{src_paris}, {src_hist}",
  layer='silver',
  output_path=silver_out
)

✅ Dataset integrated_medal_tally_1896_2024 processado com sucesso na camada silver.
